# Setup and Dependencies


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install timm efficientnet-pytorch
!pip install kaggle pandas scikit-learn opencv-python tqdm seaborn
!pip install torchmetrics
!pip install pydicom

import sys
print("✓ Dependencies installed!")
print(f"Python version: {sys.version}")

import torch
print(f"PyTorch version: {torch.__version__}")
# Use torch.version.cuda to get the CUDA toolkit version
print(f"CUDA toolkit version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Looking in indexes: https://download.pytorch.org/whl/cu118
  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=1dedae2d325f92fe8fe070ce3743e51227956cc0df053f51b686cca3e8f51335
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
Successfully built efficientnet-pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.2 MB/s eta 0:00:00
✓ Dependencies installed!
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch version: 2.10.0+cu128
CUDA toolkit version: 12.8
GPU: Tesla T4


# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directory for saving models
import os
os.makedirs('/content/drive/MyDrive/EyeShield/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/EyeShield/logs', exist_ok=True)

print("✓ Google Drive mounted!")
print("Save location: /content/drive/MyDrive/EyeShield/")

Mounted at /content/drive
✓ Google Drive mounted!
Save location: /content/drive/MyDrive/EyeShield/


#Setup Kaggle API

In [ ]:
from google.colab import files
import json
from pathlib import Path

print("Upload your kaggle.json file...")
print("Instructions:")
print("1. Go to: https://www.kaggle.com/settings/account")
print("2. Click 'Create New API Token'")
print("3. Upload the downloaded kaggle.json file")

uploaded = files.upload()

if 'kaggle.json' in uploaded:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)

    with open(kaggle_dir / 'kaggle.json', 'w') as f:
        f.write(uploaded['kaggle.json'].decode())

    os.chmod(kaggle_dir / 'kaggle.json', 0o600)
    print("✓ Kaggle API configured!")
else:
    print("⚠ kaggle.json not found. You can continue with sample data.")

Upload your kaggle.json file...
Instructions:
1. Go to: https://www.kaggle.com/settings/account
2. Click 'Create New API Token'
3. Upload the downloaded kaggle.json file


Saving kaggle.json to kaggle.json
✓ Kaggle API configured!


# Download Dataset from Kaggle

In [ ]:
import kagglehub

# Unzip if needed
import zipfile
dataset_path = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy")

for file in os.listdir(dataset_path):
    if file.endswith('.zip'):
        with zipfile.ZipFile(os.path.join(dataset_path, file), 'r') as zip_ref:
            zip_ref.extractall(dataset_path)

print(f"✓ Dataset downloaded to: {dataset_path}")
print(f"Files: {os.listdir(dataset_path)[:10]}")


Using Colab cache for faster access to the 'eyepacs-aptos-messidor-diabetic-retinopathy' dataset.
✓ Dataset downloaded to: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy
Files: ['dr_unified_v2', 'augmented_resized_V2']


# Copy Training Script

In [10]:
!wget -O /content/eyeshield_training_preprocessor.py \
  "https://raw.githubusercontent.com/dondondon22/EyeShield/refs/heads/main/eyeshield_training_preprocessor.py"

# OR if using GitHub Gist:
# !wget -O /content/eyeshield_sprint3_training.py "https://gist.githubusercontent.com/..."

print("✓ Training script downloaded!")

--2026-03-04 17:27:26--  https://raw.githubusercontent.com/dondondon22/EyeShield/refs/heads/main/eyeshield_training_preprocessor.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30692 (30K) [text/plain]
Saving to: ‘/content/eyeshield_training_preprocessor.py’

/content/eyeshield_ 100%[===================>]  29.97K  --.-KB/s    in 0.001s  

2026-03-04 17:27:26 (22.2 MB/s) - ‘/content/eyeshield_training_preprocessor.py’ saved [30692/30692]

✓ Training script downloaded!


# Prepare Dataset CSV

In [11]:
import pandas as pd
import os
from pathlib import Path

# The dataset was downloaded to `dataset_path` variable in the previous step.
# Use this variable for the root directory of the dataset.
dataset_root = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy") # Re-define for isolated execution

# Create directory for the CSV file if it doesn't exist
os.makedirs('/content/dataset', exist_ok=True)

images = []
labels = []

print(f"DEBUG: Dataset root: {dataset_root}")
print(f"DEBUG: Contents of dataset root: {os.listdir(dataset_root)}")

# The dataset contains subdirectories like 'dr_unified_v2' and 'augmented_resized_V2'
# We need to iterate through these to find the actual image folders (0, 1, 2, 3, 4)
for sub_dir in os.listdir(dataset_root):
    current_dataset_path = os.path.join(dataset_root, sub_dir)
    if os.path.isdir(current_dataset_path):
        print(f"DEBUG:   Processing subdirectory: {current_dataset_path}")
        # Look for 'train', 'test', 'validation' directories
        for data_split in os.listdir(current_dataset_path):
            data_split_path = os.path.join(current_dataset_path, data_split)
            if os.path.isdir(data_split_path):
                print(f"DEBUG:     Processing data split: {data_split_path}")
                # Now look for the actual DR level directories (0, 1, 2, 3, 4)
                for dr_level in os.listdir(data_split_path):
                    level_dir = os.path.join(data_split_path, dr_level)
                    if os.path.isdir(level_dir):
                        try:
                            level_int = int(dr_level)
                            # print(f"DEBUG:       Processing DR level directory: {level_dir}")
                            found_images_in_level = False
                            for img_file in os.listdir(level_dir):
                                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                                    full_img_path = os.path.join(level_dir, img_file)
                                    images.append(os.path.relpath(full_img_path, dataset_root))
                                    labels.append(level_int)
                                    found_images_in_level = True
                            if not found_images_in_level:
                                print(f"DEBUG:         No images found in {level_dir} or they don't match extensions.")
                        except ValueError:
                            print(f"DEBUG:       Skipping non-integer directory: {level_dir}")
                    else:
                        print(f"DEBUG:       {level_dir} is not a directory. Skipping.")
            else:
                print(f"DEBUG:     {data_split_path} is not a directory. Skipping.")
    else:
        print(f"DEBUG:   {current_dataset_path} is not a directory. Skipping.")

df = pd.DataFrame({
    'image_path': images,
    'diagnosis': labels
})

df.to_csv('/content/dataset/labels.csv', index=False)

print(f"✓ Dataset CSV created!")
print(f"Total images: {len(df)}")
print(f"Class distribution:\n{df['diagnosis'].value_counts().sort_index()}")
print(f"\nSample:\n{df.head()}")

Using Colab cache for faster access to the 'eyepacs-aptos-messidor-diabetic-retinopathy' dataset.
DEBUG: Dataset root: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy
DEBUG: Contents of dataset root: ['dr_unified_v2', 'augmented_resized_V2']
DEBUG:   Processing subdirectory: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2
DEBUG:     Processing data split: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2
DEBUG:       Skipping non-integer directory: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2/val
DEBUG:       Skipping non-integer directory: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2/test
DEBUG:       Skipping non-integer directory: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2/train
DEBUG:   Processing subdirectory: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/augmented_resized_V2
D

# Modify Config and Run Training

In [12]:
# Read the training script
with open('/content/eyeshield_training_preprocessor.py', 'r') as f:
    training_code = f.read()

# Modify paths in the Config class (if needed)
modified_code = training_code.replace(
    "CHECKPOINT_DIR = './checkpoints'",
    "CHECKPOINT_DIR = '/content/drive/MyDrive/EyeShield/checkpoints'"
)

modified_code = modified_code.replace(
    "LOG_DIR = './logs'",
    "LOG_DIR = '/content/drive/MyDrive/EyeShield/logs'"
)

# Save modified script
with open('/content/eyeshield_training_preprocessor_modified.py', 'w') as f:
    f.write(modified_code)

print("✓ Configuration updated!")
print("Ready to start training...")

✓ Configuration updated!
Ready to start training...


#  Run Training

In [13]:
# Execute the full training pipeline

%cd /content

# Import required modules
import sys
sys.path.insert(0, '/content')

# Execute training
exec(open('/content/eyeshield_training_preprocessor_modified.py').read())
main()

/content
Device: cuda
CUDA Available: True
GPU: Tesla T4

EyeShield: DR Classification Model Training (Sprint 3)
Using Your Image Preprocessor (No CLAHE)
Configuration:
  - Num Classes: 5
  - Target Preprocessing Size: (512, 512)
  - Input Size: (512, 512)
  - Batch Size: 16
  - Num Epochs: 50
  - Learning Rate: 0.0001
  - EDL KL Weight: 0.1
  - Quality Check: True

Initializing image preprocessor...
✓ Image preprocessor initialized
  - Target size: (512, 512)
  - Quality assessment: Enabled

Loading dataset from CSV...
Using Colab cache for faster access to the 'eyepacs-aptos-messidor-diabetic-retinopathy' dataset.
✓ Loaded 143669 images from dataset
  - Class distribution:
diagnosis
0    68953
1    22172
2    30221
3     9914
4    12409
Name: count, dtype: int64

Data split:
  - Train: 100568 images
  - Val: 21550 images
  - Test: 21551 images

Initializing model...
Total Parameters: 11,090,989
Trainable Parameters: 11,090,989


Starting Training: EfficientNet-B3 + EDL for DR Classif

Validation: 100%|██████████| 1347/1347 [08:39<00:00,  2.59it/s]



Epoch 1/50
Train Loss: 0.2781 | Train Acc: 0.8256
Val Loss: 2.6932 | Val Acc: 0.3865 | ECE: 0.5262


TypeError: cannot pickle 'mappingproxy' object

# Monitor Training (Optional)

In [ ]:
# Use tensorboard to monitor training in real-time

# %load_ext tensorboard
# %tensorboard --logdir /content/logs

In [ ]:
# Download Results